# Hotstart Component Tutorial

This tutorial covers the **Hotstart** component in `rompy-xbeach`, which enables
initializing XBeach simulations from a previous simulation state.

## What You'll Learn

1. What hotstart files are and when to use them
2. Writing hotstart files during a simulation
3. Reading hotstart files to initialize a new simulation
4. Workflow for chained simulations

## Prerequisites

- Familiarity with the `Config` class
- Understanding of XBeach simulation workflow

## Helper Functions

In [1]:
def print_params(lines: str, filename: str = "params.txt"):
    """Display parameters in a text-editor style format."""
    border = "─" * 60
    print(f"┌{border}┐")
    print(f"│ {filename:<58} │")
    print(f"├{border}┤")
    for line in lines.strip().split("\n"):
        print(f"│ {line:<58} │")
    print(f"└{border}┘")


def show_params(obj, destdir=None):
    """Show parameters that would be written to params.txt."""
    import tempfile

    if destdir is None:
        destdir = tempfile.mkdtemp()
    params = obj.get(destdir=destdir)
    lines = "\n".join(f"{k} = {v}" for k, v in params.items())
    print_params(lines)
    return params


def print_warning(text: str):
    """Display a warning message with styling."""
    border = "─" * 60
    print(f"\033[93m┌{border}┐\033[0m")
    print(f"\033[93m│ ⚠ WARNING{' ' * 49}│\033[0m")
    print(f"\033[93m├{border}┤\033[0m")
    for line in text.strip().split("\n"):
        print(f"\033[93m│ {line:<58} │\033[0m")
    print(f"\033[93m└{border}┘\033[0m")

---
## 1. Introduction to Hotstart

### What is Hotstart?

Hotstart is a feature that allows XBeach to:
- **Save** the simulation state at specified intervals during a run
- **Load** a saved state to initialize a subsequent simulation

This is useful for:
- **Avoiding spin-up time**: Skip the initial transient period
- **Chaining simulations**: Run long simulations in segments
- **Scenario analysis**: Branch from a common initial state
- **Recovery**: Resume from a checkpoint if a simulation fails

### Hotstart Files

XBeach writes hotstart files with the naming convention:
```
hotstart_{varname}{fileno:06d}.dat
```

For example, with `hotstartfileno=3`:
- `hotstart_zs000003.dat` - Water surface elevation
- `hotstart_zb000003.dat` - Bed level
- `hotstart_uu000003.dat` - Velocity in x-direction
- `hotstart_vv000003.dat` - Velocity in y-direction

Additional files may be written depending on simulation options (groundwater,
wave energy, sediment concentration, etc.).

---
## 2. Writing Hotstart Files

To write hotstart files during a simulation, use the `Output` component's
`writehotstart` and `tinth` parameters.

In [2]:
from rompy_xbeach.components.output import Output

# Enable hotstart output at the end of simulation only
output_hotstart_end = Output(
    writehotstart=True,
    # tinth not set or 0 means write only at simulation end
)

show_params(output_hotstart_end)

┌────────────────────────────────────────────────────────────┐
│ params.txt                                                 │
├────────────────────────────────────────────────────────────┤
│ outputformat = netcdf                                      │
│ writehotstart = True                                       │
└────────────────────────────────────────────────────────────┘


{'outputformat': 'netcdf', 'writehotstart': True}

### Writing at Regular Intervals

Set `tinth` to write hotstart files at regular intervals:

In [3]:
# Write hotstart files every hour (3600 seconds)
output_hotstart_hourly = Output(
    writehotstart=True,
    tinth=3600.0,  # Hotstart output interval in seconds
)

show_params(output_hotstart_hourly)

┌────────────────────────────────────────────────────────────┐
│ params.txt                                                 │
├────────────────────────────────────────────────────────────┤
│ outputformat = netcdf                                      │
│ tinth = 3600.0                                             │
│ writehotstart = True                                       │
└────────────────────────────────────────────────────────────┘


{'outputformat': 'netcdf', 'tinth': 3600.0, 'writehotstart': True}

Each time a hotstart is written, the file number increments. So after a 6-hour
simulation with `tinth=3600`:
- Hour 1: `hotstart_*000001.dat`
- Hour 2: `hotstart_*000002.dat`
- ...
- Hour 6: `hotstart_*000006.dat`
- End: `hotstart_*000007.dat` (if simulation doesn't end exactly on interval)

---
## 3. Reading Hotstart Files

To initialize a simulation from hotstart files, use the `Hotstart` component.

In [4]:
from rompy_xbeach.components.hotstart import Hotstart

# Basic hotstart - files must exist in run directory
hotstart_basic = Hotstart()

show_params(hotstart_basic)

┌────────────────────────────────────────────────────────────┐
│ params.txt                                                 │
├────────────────────────────────────────────────────────────┤
│ hotstart = 1                                               │
│ hotstartfileno = 0                                         │
└────────────────────────────────────────────────────────────┘


{'hotstart': 1, 'hotstartfileno': 0}

### Selecting a Specific Hotstart File

Use `hotstartfileno` to select which snapshot to use:

In [5]:
# Initialize from the 3rd hotstart snapshot
hotstart_from_3 = Hotstart(hotstartfileno=3)

show_params(hotstart_from_3)

┌────────────────────────────────────────────────────────────┐
│ params.txt                                                 │
├────────────────────────────────────────────────────────────┤
│ hotstart = 1                                               │
│ hotstartfileno = 3                                         │
└────────────────────────────────────────────────────────────┘


{'hotstart': 1, 'hotstartfileno': 3}

### Copying Files from a Previous Run

Use `previous_run` to automatically copy hotstart files from another directory.
This is useful when chaining simulations across different run directories.

```python
from rompy_xbeach.types import XBeachDirectoryBlob

# Copy hotstart files from a previous simulation
hotstart_with_source = Hotstart(
    hotstartfileno=5,
    previous_run=XBeachDirectoryBlob(source="/path/to/previous/run"),
)

# When get() is called, it will copy hotstart_*000005.dat files to destdir
params = hotstart_with_source.get(destdir="/path/to/new/run")
```

The `XBeachDirectoryBlob` will copy files matching the pattern
`hotstart_*000005.dat` from the source directory to the destination.

---
## 4. Integration with Config

### Writing Hotstart (First Simulation)

In [6]:
from rompy_xbeach.config import Config
from rompy_xbeach.components.output import Output

# First simulation - write hotstart files
# config_first = Config(
#     grid=grid,
#     bathy=bathy,
#     input=data_interface,
#     output=Output(
#         writehotstart=True,
#         tinth=3600.0,  # Write every hour
#         outputformat="netcdf",
#         tintg=600.0,
#     ),
# )

print("First simulation config would include:")
print("  output.writehotstart = True")
print("  output.tinth = 3600.0")

First simulation config would include:
  output.writehotstart = True
  output.tinth = 3600.0


### Reading Hotstart (Continuation Simulation)

In [7]:
# Continuation simulation - read from previous run
# config_continuation = Config(
#     grid=grid,
#     bathy=bathy,
#     input=data_interface,
#     hotstart=Hotstart(
#         hotstartfileno=6,  # Use 6th snapshot (hour 6)
#         previous_run=XBeachDirectoryBlob(source="/path/to/first/run"),
#     ),
#     output=Output(
#         writehotstart=True,  # Continue writing hotstart files
#         tinth=3600.0,
#         outputformat="netcdf",
#     ),
# )

print("Continuation simulation config would include:")
print("  hotstart.hotstart = 1")
print("  hotstart.hotstartfileno = 6")

Continuation simulation config would include:
  hotstart.hotstart = 1
  hotstart.hotstartfileno = 6


### Simple Boolean Hotstart

For simple cases where hotstart files already exist in the run directory:

In [8]:
# config_simple = Config(
#     grid=grid,
#     bathy=bathy,
#     input=data_interface,
#     hotstart=True,  # Enable hotstart with default file number (0)
# )

print("Simple hotstart: hotstart=True")
print("  - Expects hotstart_*000000.dat files in run directory")

Simple hotstart: hotstart=True
  - Expects hotstart_*000000.dat files in run directory


---
## 5. Workflow Example

A typical workflow for chained simulations:

```
┌─────────────────────────────────────────────────────────────┐
│  Simulation 1: Spin-up (e.g., 24 hours)                     │
│  - writehotstart=True, tinth=3600                           │
│  - Writes: hotstart_*000001.dat ... hotstart_*000024.dat    │
└─────────────────────────────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  Simulation 2: Storm event (e.g., 12 hours)                 │
│  - hotstart=Hotstart(hotstartfileno=24, previous_run=...)   │
│  - Reads: hotstart_*000024.dat from Simulation 1            │
│  - writehotstart=True for potential continuation            │
└─────────────────────────────────────────────────────────────┘
                             │
                             ▼
┌─────────────────────────────────────────────────────────────┐
│  Simulation 3: Recovery (e.g., 48 hours)                    │
│  - hotstart=Hotstart(hotstartfileno=12, previous_run=...)   │
│  - Reads: hotstart_*000012.dat from Simulation 2            │
└─────────────────────────────────────────────────────────────┘
```

---
## 6. Important Considerations

### Grid Consistency

**Critical**: The grid dimensions must be identical between simulations.
Hotstart files contain arrays that match the original grid size.

In [9]:
print_warning(
    "Hotstart files are grid-specific!\n"
    "The continuation simulation must use the same grid\n"
    "dimensions (nx, ny) as the original simulation."
)

┌────────────────────────────────────────────────────────────┐
│ ⚠ WARNING                                                 │
├────────────────────────────────────────────────────────────┤
│ Hotstart files are grid-specific!                          │
│ The continuation simulation must use the same grid         │
│ dimensions (nx, ny) as the original simulation.            │
└────────────────────────────────────────────────────────────┘


### File Number Range

The `hotstartfileno` parameter accepts values from 0 to 999:

In [10]:
try:
    invalid_hotstart = Hotstart(hotstartfileno=1000)
except Exception as e:
    print_warning(f"Validation error:\n{e}")

┌────────────────────────────────────────────────────────────┐
│ ⚠ WARNING                                                 │
├────────────────────────────────────────────────────────────┤
│ Validation error:                                          │
│ 1 validation error for Hotstart                            │
│ hotstartfileno                                             │
│   Input should be less than or equal to 999 [type=less_than_equal, input_value=1000, input_type=int] │
│     For further information visit https://errors.pydantic.dev/2.12/v/less_than_equal │
└────────────────────────────────────────────────────────────┘


### Manual Mode Validation

When using `mpiboundary='man'` for MPI, both `mmpi` and `nmpi` must be specified.
Similarly, hotstart requires consistent MPI decomposition between runs.

---
## 7. Summary

### Key Parameters

| Parameter | Location | Description |
|-----------|----------|-------------|
| `writehotstart` | `Output` | Enable writing hotstart files |
| `tinth` | `Output` | Interval for hotstart output (s) |
| `hotstart` | `Config` | Enable reading hotstart files |
| `hotstartfileno` | `Hotstart` | Which snapshot to read (0-999) |
| `previous_run` | `Hotstart` | Source directory for hotstart files |

### Typical Usage

```python
# Writing hotstart files
output = Output(writehotstart=True, tinth=3600.0)

# Reading hotstart files
hotstart = Hotstart(hotstartfileno=24, previous_run=XBeachDirectoryBlob(...))

# Simple enable (files in run directory)
config = Config(..., hotstart=True)
```

### Next Steps

- See the **MPI Component** tutorial for parallel execution
- See the **Output Component** tutorial for output configuration details